# AFSP regenerated under the corrected centroid (val)

`afsp_full` and `peft_afsp` were generated with exemplars chosen by a reranker reading
the pre-case-fold centroid. AFSP calls `register_band_distance` against that centroid at
selection time, so the broken marker counter is baked into *which exemplars each prompt
got*, not only into how the resulting translations were later measured. Rescoring the
existing outputs measures old choices more accurately; it does not recover the method.

This session regenerates both conditions as `afsp_full_casefix` and `peft_afsp_casefix`
at the frozen operating point -- k=8, lambda_style=0.75, beta=0.3, sigma=1.0, greedy,
seed 42. Nothing is retuned: the `--score-only` pass already confirmed both picks survive
the fix. The only thing that changes is which centroid the reranker reads.

Budget: ~1890 s per 1323-segment pass, two passes, ~1.5 h with model load. No paid call.

---
## 1 — Host, working tree, disk

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version --format=csv

In [ ]:
from pathlib import Path

if not Path('manage.py').exists():
    if not Path('Style-Aware-MT/manage.py').exists():
        !git clone https://github.com/prnamhr/Style-Aware-MT.git
    %cd Style-Aware-MT
!git pull --ff-only
!git rev-parse --short HEAD

In [ ]:
# %pip installs into the kernel; !pip may not.
%pip install -r requirements.txt

In [ ]:
# torch 2.12 breaks the pinned-torch ABI these three ship against; the pipeline is text-only.
%pip uninstall -q -y torchvision torchaudio torchcodec

In [ ]:
import torch

cap = torch.cuda.get_device_capability(0)
print(f'{torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}',
      f'torch {torch.__version__} / cuda {torch.version.cuda}')
assert torch.cuda.is_bf16_supported(), 'bf16 unsupported; the frozen base is not quantized'

---
## 2 — Run parameters

Both conditions share one `retrieval:`/`afsp:`/`prompt:` block; the configs differ only in
`generator.adapter_path`. Exemplar selection is therefore identical between them, which is
why section 4 probes it once.

In [ ]:
import json
import os
import subprocess
import sys
import time
from datetime import datetime, timedelta, timezone

import yaml

SPLIT = 'val'
CASEFIX_COMMIT = '141c532'          # where the marker regex became case-folding
CENTROID = Path('results/stylometrics_centroid.json')
CORRECTED_FINGERPRINT = 'fd5aec8d69454b02'

# (condition to run, config, output stem). --condition stays a known rung; --out-name
# is what keeps the corrected pass beside the old one instead of overwriting it.
RUNS = [
    ('afsp_full', Path('configs/base_qwen.yaml'), 'afsp_full_casefix'),
    ('peft_afsp', Path('configs/peft_afsp.yaml'), 'peft_afsp_casefix'),
]

CFGS = {c: yaml.safe_load(p.read_text(encoding='utf-8')) for c, p, _ in RUNS}
CFG = CFGS['afsp_full']
RETR, AFSP, PROMPT = CFG['retrieval'], CFG['afsp'], CFG['prompt']

for blk in ('retrieval', 'afsp', 'prompt', 'data'):
    assert CFGS['afsp_full'][blk] == CFGS['peft_afsp'][blk], f'{blk} differs across the two configs'

for cond, path, _ in RUNS:
    gen = CFGS[cond]['generator']
    assert gen['model'] == 'Qwen/Qwen2.5-7B-Instruct', gen['model']
    assert (gen['temperature'], gen['top_p']) == (0.0, 1.0), 'not the locked greedy decoding'
    assert (gen['max_tokens'], gen['seed']) == (1024, 42), gen
    assert gen['dtype'] == 'bfloat16' and gen['load_in_4bit'] is False, 'quantizing redefines the base'
    assert CFGS[cond]['data']['eval_file'] == f'data/splits/{SPLIT}.jsonl', CFGS[cond]['data']

# The frozen operating point (DEVLOG 2026-07-23), confirmed unmoved by afsp_sweep --score-only.
assert RETR['k'] == 8, RETR
assert (AFSP['beta'], AFSP['lambda_style']) == (0.3, 0.75), AFSP
assert (AFSP['style_objective'], AFSP['style_target_sigma']) == ('bandpass', 1.0), AFSP
assert AFSP['centroid_file'] == str(CENTROID), AFSP['centroid_file']

ADAPTER = Path(CFGS['peft_afsp']['generator']['adapter_path'])
assert str(ADAPTER) == 'models/peft_lora_r32_lr2e-4/checkpoint-1358', ADAPTER

DEADLINE = datetime.now(timezone.utc) + timedelta(hours=3)
print(f'k={RETR["k"]} lambda={AFSP["lambda_style"]} beta={AFSP["beta"]} '
      f'sigma={AFSP["style_target_sigma"]}  ->  {[n for _, _, n in RUNS]}')

In [ ]:
from src.eval.stylometrics import fingerprint

CENTROID_NEW = json.loads(CENTROID.read_text(encoding='utf-8'))
FP = fingerprint(CENTROID_NEW)
assert FP == CORRECTED_FINGERPRINT, (
    f'{CENTROID} fingerprints to {FP}, not the corrected {CORRECTED_FINGERPRINT}. '
    f'This run would bake a third centroid into the outputs.')

mr = CENTROID_NEW['features'].index('marker_rate')
print(f'centroid {FP}  marker_rate mean={CENTROID_NEW["mean"][mr]:.6f} '
      f'std={CENTROID_NEW["std"][mr]:.6f}')

In [ ]:
import getpass
import logging

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN: ')
for var in ('OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY'):
    assert not os.environ.get(var), f'{var} is set; this session makes no paid call'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
logging.getLogger('httpx').setLevel(logging.WARNING)
print('HF_TOKEN set, no rater keys present')

---
## 3 — Weights and the index, before the GPU is touched

Network first. Every download that happens after the base is resident is a GPU-hour spent
waiting on bandwidth.

In [ ]:
import hashlib

from huggingface_hub import snapshot_download

HF_REPO = 'prnamhr/style-aware-mt-models'
ADAPTER_FILES = ('adapter_config.json', 'adapter_model.safetensors')

if not all((ADAPTER / f).exists() for f in ADAPTER_FILES):
    snapshot_download(HF_REPO, local_dir='.', token=os.environ['HF_TOKEN'],
                      allow_patterns=[f'{ADAPTER}/{f}' for f in ADAPTER_FILES])

conf = json.loads((ADAPTER / 'adapter_config.json').read_text(encoding='utf-8'))
assert conf['r'] == 32 and conf['lora_alpha'] == 64, conf
assert conf['base_model_name_or_path'].endswith(CFG['generator']['model'].split('/')[-1]), conf
ADAPTER_SHA = hashlib.sha256((ADAPTER / 'adapter_model.safetensors').read_bytes()).hexdigest()
print(f'{ADAPTER}  r={conf["r"]} alpha={conf["lora_alpha"]}  {ADAPTER_SHA[:12]}')

In [ ]:
t0 = time.perf_counter()
snapshot_download(CFG['generator']['model'],
                  allow_patterns=['*.json', '*.safetensors', '*.txt', '*.jinja'],
                  token=os.environ['HF_TOKEN'], max_workers=8)
print(f"base cached in {(time.perf_counter() - t0) / 60:.1f} min")

In [ ]:
INDEX = Path(RETR['index_dir'])
INDEX_FILES = ('embeddings.npy', 'pairs.jsonl', 'meta.json')
INDEX_REBUILT = not all((INDEX / f).exists() for f in INDEX_FILES)

if INDEX_REBUILT:
    print('index missing -- rebuilding; exemplar selection may differ from the reported AFSP run')
    !python3 manage.py build_index --config configs/base_qwen.yaml

INDEX_SHA = {f: hashlib.sha256((INDEX / f).read_bytes()).hexdigest() for f in INDEX_FILES}
meta = json.loads((INDEX / 'meta.json').read_text(encoding='utf-8'))
assert meta['embed_model'] == RETR['embed_model'] and meta['indexed_side'] == 'source', meta
assert meta['n_passages'] == 10860, meta
print(f'index {"rebuilt" if INDEX_REBUILT else "transferred"}, {meta["n_passages"]} passages')
for f, digest in INDEX_SHA.items():
    print(f'  {f:16s} {digest[:12]}')

---
## 4 — The gate: does the corrected centroid move exemplar selection?

The premise of this session is that it does. Measure it before spending the GPU rather
than asserting it afterwards: build the reranker twice over the same index, once with the
centroid as it was at `141c532^` and once as it is now, and compare the selected exemplar
sets segment by segment. Retrieval only -- no generation, no adapter.

If selection turns out to be unchanged, regeneration buys nothing and the run should be
abandoned in favour of the rescored artifacts already committed.

In [ ]:
from src.retrieval.afsp import AFSPRetriever
from src.retrieval.retrieve import RetrievalIndex

ROWS = [json.loads(x) for x in Path(CFG['data']['eval_file']).open(encoding='utf-8') if x.strip()]
VAL_SRC = [r['input'] for r in ROWS]
print(f'{len(ROWS)} {SPLIT} segments')

CENTROID_OLD = json.loads(subprocess.run(
    ['git', 'show', f'{CASEFIX_COMMIT}^:results/stylometrics_centroid.json'],
    capture_output=True, text=True, check=True).stdout)
FP_OLD = fingerprint(CENTROID_OLD)
assert FP_OLD != FP, 'the two centroids are the same file; there is nothing to regenerate'
print(f'old {FP_OLD}  marker_rate mean={CENTROID_OLD["mean"][mr]:.6f} '
      f'std={CENTROID_OLD["std"][mr]:.6f}')

In [ ]:
index = RetrievalIndex(RETR['index_dir'], embed_model=RETR['embed_model'])

def reranker(centroid):
    """Exactly how src.infer.run._select_afsp builds it for a rerank condition."""
    return AFSPRetriever(
        index, centroid, index_dir=RETR['index_dir'],
        beta=AFSP['beta'], knn_hubness=AFSP['knn_hubness'], pool_mult=AFSP['pool_mult'],
        lambda_style=AFSP['lambda_style'], style_objective=AFSP['style_objective'],
        style_target_sigma=AFSP['style_target_sigma'],
        style_register_direction=AFSP['style_register_direction'])

t0 = time.perf_counter()
SEL_OLD = reranker(CENTROID_OLD).select(VAL_SRC, k=RETR['k'])
SEL_NEW = reranker(CENTROID_NEW).select(VAL_SRC, k=RETR['k'])
print(f'both selections in {time.perf_counter() - t0:.0f}s')

In [ ]:
old_sets = [tuple(e['input'] for e in row) for row in SEL_OLD]
new_sets = [tuple(e['input'] for e in row) for row in SEL_NEW]

changed_set = [i for i, (a, b) in enumerate(zip(old_sets, new_sets)) if set(a) != set(b)]
changed_order = [i for i, (a, b) in enumerate(zip(old_sets, new_sets))
                 if set(a) == set(b) and a != b]
overlap = [len(set(a) & set(b)) for a, b in zip(old_sets, new_sets)]

SELECTION_DIVERGENCE = {
    'n_segments': len(ROWS),
    'k': RETR['k'],
    'changed_set': len(changed_set),
    'changed_order_only': len(changed_order),
    'identical': len(ROWS) - len(changed_set) - len(changed_order),
    'mean_exemplars_shared': round(sum(overlap) / len(overlap), 3),
    'centroid_old': FP_OLD,
    'centroid_new': FP,
}
for key, val in SELECTION_DIVERGENCE.items():
    print(f'  {key:24s} {val}')

In [ ]:
share = SELECTION_DIVERGENCE['changed_set'] / len(ROWS)
if SELECTION_DIVERGENCE['changed_set'] == 0 and SELECTION_DIVERGENCE['changed_order_only'] == 0:
    print('\nSelection is unchanged. Regeneration buys nothing -- stop here and keep the '
          'rescored artifacts.')
else:
    print(f'\n{share:.1%} of prompts get a different exemplar set under the corrected '
          f'centroid, {SELECTION_DIVERGENCE["changed_order_only"]} more get the same set in a '
          f'different order. Rescoring cannot reach this. Section 6 may run.')

---
## 5 — Throughput and fit

In [ ]:
from src.infer.run import _load_configured_glossary, build_fewshot_user, make_client, order_exemplars

STYLE = Path(PROMPT['style_instruction_file']).read_text(encoding='utf-8')
GLOSSARY = _load_configured_glossary(CFG)

PROBE_N = 8
probe = [build_fewshot_user(s, order_exemplars(ex, PROMPT['ordering']), GLOSSARY)
         for s, ex in zip(VAL_SRC[:PROBE_N], SEL_NEW[:PROBE_N])]

# The adapter pass is the slower of the two, so it bounds both; probing it here also
# forces PeftModel.from_pretrained before section 6 rather than 25 minutes into it.
PROBE_GEN = CFGS['peft_afsp']['generator']
t0 = time.perf_counter()
client = make_client(PROBE_GEN)
load_s = time.perf_counter() - t0

t0 = time.perf_counter()
for user in probe:
    client.complete(STYLE, user)
seg_s = (time.perf_counter() - t0) / PROBE_N

print(f'{load_s:.0f}s load with {Path(PROBE_GEN["adapter_path"]).name}, '
      f'{seg_s:.2f}s per segment at k={RETR["k"]}')
print(f'{torch.cuda.max_memory_allocated() / 2**30:.1f} GiB peak')

In [ ]:
# seg_s came off the adapter, so this is an upper bound on the base pass too.
pass_h = len(ROWS) * seg_s / 3600
left_h = (DEADLINE - datetime.now(timezone.utc)).total_seconds() / 3600
print(f'{pass_h:.2f} h per pass, {pass_h * len(RUNS):.1f} h for {len(RUNS)}, '
      f'{left_h:.1f} h left of the booking')

if pass_h * len(RUNS) > 0.9 * left_h:
    print('\nDoes not fit.')
else:
    print('\nFits. Section 6 may start.')

In [ ]:
# The probe held the base and the adapter; free both before manage.py loads its own.
del client
torch.cuda.empty_cache()

---
## 6 — Generation

In [ ]:
TIMING = {}
for cond, config, name in RUNS:
    t0 = time.perf_counter()
    r = subprocess.run([sys.executable, 'manage.py', 'infer', '--condition', cond,
                        '--config', str(config), '--out-name', name], check=False)
    assert r.returncode == 0, f'{name} exited {r.returncode}'
    TIMING[name] = {'seconds': round(time.perf_counter() - t0, 1),
                    'finished': datetime.now(timezone.utc).isoformat()}
    print(f'{name}: {TIMING[name]["seconds"] / 60:.1f} min')

---
## 7 — The outputs

`--out-name` changes the filename, not the `condition` field written into each row: the
rows still say `afsp_full`/`peft_afsp`, which is correct -- the method is the same, only
the centroid it read has been fixed.

In [ ]:
for cond, _, name in RUNS:
    path = Path(f'outputs/{name}_{SPLIT}.jsonl')
    rows = [json.loads(x) for x in path.open(encoding='utf-8') if x.strip()]
    assert len(rows) == len(ROWS), f'{name}: {len(rows)} rows, expected {len(ROWS)}'
    assert [r['input'] for r in rows] == VAL_SRC, f'{name}: source order differs from {SPLIT}.jsonl'
    assert all(r['condition'] == cond for r in rows), f'{name}: mislabelled rows'
    blank = [i for i, r in enumerate(rows) if not r['prediction'].strip()]
    old = Path(f'outputs/{cond}_{SPLIT}.jsonl')
    moved = sum(1 for a, b in zip(rows, [json.loads(x) for x in old.open(encoding='utf-8') if x.strip()])
                if a['prediction'] != b['prediction']) if old.exists() else None
    print(f'{name}: {len(rows)} rows, {len(blank)} blank {blank[:5]}, '
          f'{moved} translations differ from {cond}')

In [ ]:
# The corrected pass must record the corrected centroid, not inherit the old provenance.
for _, _, name in RUNS:
    usage = json.loads(Path(f'outputs/{name}_{SPLIT}_usage.json').read_text(encoding='utf-8'))
    prov = usage['provenance']
    assert prov['centroid']['fingerprint'] == CORRECTED_FINGERPRINT, prov
    assert prov['lambda_style'] == AFSP['lambda_style'] and prov['k'] == RETR['k'], prov
    print(f'{name}: {json.dumps(prov)}')

---
## 8 — Scoring on the host

Surface metrics and the register interval, both of which run in this kernel. Every result
written here carries the `fd5aec8d` fingerprint automatically, so it lands self-identifying
beside the rest of `results/`.

COMET and the two `heldout_decomp` decompositions are not run here. COMET pins
transformers 4.57.6 and numpy 1.26.4 against this environment's 5.12.1 and 2.4.1, so it
needs a venv of its own, and `heldout_decomp` silently drops its adequacy columns for any
condition COMET has not scored yet. Both belong to the local pass in §11.

In [ ]:
NEW_CONDS = [n for _, _, n in RUNS]
# Each corrected stem beside the run it supersedes, plus the plain-retrieval control.
EVAL_CONDS = ' '.join([c for cond, _, name in RUNS for c in (cond, name)] + ['peft_knn'])
!python3 manage.py eval --split {SPLIT} --conditions {EVAL_CONDS}

In [ ]:
CI_CONDS = ' '.join(['knn_fewshot', 'peft', 'peft_knn']
                    + [c for cond, _, name in RUNS for c in (cond, name)])
CI_PATH = f'results/stylometrics_ci_casefix_{SPLIT}.json'
!python3 manage.py stylometrics_ci --split {SPLIT} --conditions {CI_CONDS} --results_path {CI_PATH}

In [ ]:
# What the regeneration actually bought, on the register axis.
ci = json.loads(Path(CI_PATH).read_text(encoding='utf-8'))
assert ci['centroid']['fingerprint'] == CORRECTED_FINGERPRINT, ci['centroid']
# stylometrics_ci skips a condition whose file is missing rather than failing.
missing = [n for n in NEW_CONDS if n not in ci['conditions']]
assert not missing, f'{missing} were skipped; the report does not contain the new run'
print(f"{'condition':22s} {'stylo_dist':>10s}  rank")
for cond in ci['ranking']:
    print(f'{cond:22s} {ci["cells"][cond]["stylo_dist"]:>10.4f}  {ci["cells"][cond]["rank"]}')

print()
for pair in (('afsp_full', 'afsp_full_casefix'), ('peft_afsp', 'peft_afsp_casefix')):
    hit = [p for p in ci['paired_all'] if {p['a'], p['b']} == set(pair)]
    for p in hit:
        print(f'{p["a"]} - {p["b"]}: {p["diff"]:+.4f} [{p["ci_low"]:+.4f}, {p["ci_high"]:+.4f}] '
              f'p={p["p_value"]:.4f}')

---
## 9 — Manifest and bundle

In [ ]:
import platform

import peft as peft_lib
import transformers

MANIFEST = {
    'purpose': 'AFSP regenerated with exemplars reranked against the corrected centroid',
    'runs': [{'condition': c, 'config': str(p), 'output_name': n} for c, p, n in RUNS],
    'split': SPLIT,
    'supersedes': ['afsp_full', 'peft_afsp'],
    'centroid': {'path': str(CENTROID), 'fingerprint': FP, 'superseded_fingerprint': FP_OLD,
                 'casefix_commit': CASEFIX_COMMIT},
    'selection_divergence': SELECTION_DIVERGENCE,
    'generator': {c: CFGS[c]['generator'] for c, _, _ in RUNS},
    'retrieval': {'k': RETR['k'], 'embed_model': RETR['embed_model'], 'index_dir': str(INDEX)},
    'afsp': {k: AFSP[k] for k in ('beta', 'lambda_style', 'style_objective',
                                  'style_target_sigma', 'pool_mult', 'knn_hubness')},
    'adapter': {'path': str(ADAPTER), 'sha256': ADAPTER_SHA,
                'r': conf['r'], 'lora_alpha': conf['lora_alpha']},
    'index': {'rebuilt_here': INDEX_REBUILT, 'sha256': INDEX_SHA, 'meta': meta},
    'timing': TIMING,
    'commit': subprocess.run(['git', 'rev-parse', 'HEAD'],
                             capture_output=True, text=True).stdout.strip(),
    'versions': {
        'device': torch.cuda.get_device_name(0),
        'torch': torch.__version__,
        'cuda': torch.version.cuda,
        'transformers': transformers.__version__,
        'peft': peft_lib.__version__,
        'python': platform.python_version(),
    },
}
out = Path('outputs/afsp_casefix_manifest.json')
out.write_text(json.dumps(MANIFEST, indent=2) + '\n', encoding='utf-8')
print(out, out.stat().st_size, 'bytes')

In [ ]:
import zipfile

# Everything the local pass needs; the host keeps nothing.
BUNDLE = Path(f'afsp_casefix_{SPLIT}.zip')
FILES = ([Path(f'outputs/{n}_{SPLIT}.jsonl') for n in NEW_CONDS]
         + [Path(f'outputs/{n}_{SPLIT}_usage.json') for n in NEW_CONDS]
         + [out, Path(CI_PATH)])
with zipfile.ZipFile(BUNDLE, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in FILES:
        assert f.exists(), f
        z.write(f, str(f))
print(f'{BUNDLE}  {BUNDLE.stat().st_size / 1e6:.1f} MB  ({len(FILES)} files)')

---
## 10 — Seal

In [ ]:
# 1. Nothing was spent.
for _, _, name in RUNS:
    usage = json.loads(Path(f'outputs/{name}_{SPLIT}_usage.json').read_text(encoding='utf-8'))
    assert usage.get('cost_usd', 0.0) == 0.0, usage
    print(f'{name}: {usage["calls"]} calls, ${usage.get("cost_usd", 0.0):.2f}')

# 2. The test split was not touched.
assert not list(Path('outputs').glob('*_test.jsonl')), 'a test-split output exists'
assert not list(Path('results').glob('*_test.json')), 'a test-split result exists'

# 3. No rater key was ever present in this session.
for var in ('OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY'):
    assert not os.environ.get(var), var

# 4. The superseded outputs are untouched -- this session adds, it does not overwrite.
!git status --short outputs/afsp_full_{SPLIT}.jsonl outputs/peft_afsp_{SPLIT}.jsonl

print('\nsealed: 0 paid calls, test split untouched, no rater key present')

In [ ]:
!git status --short outputs results

---
## 11 — The local pass

Unzip the bundle over the working tree at home, keeping the paths it stores, then score the
two remaining artifacts:

```bash
source .venv-comet/bin/activate
python manage.py comet --split val --conditions afsp_full_casefix peft_afsp_casefix
deactivate

source .venv/bin/activate
python manage.py heldout_decomp --split val --no-figure \
  --conditions knn_fewshot afsp_full afsp_full_casefix --reference knn_fewshot \
  --results_path results/heldout_decomp_casefix_afsp_vs_knn_val.json
python manage.py heldout_decomp --split val --no-figure \
  --conditions peft_knn peft_afsp peft_afsp_casefix --reference peft_knn \
  --results_path results/heldout_decomp_casefix_peft_afsp_vs_knn_val.json
```

`merge_results` keeps the twelve conditions already in `results/comet_val.json`, so the four
references these two decompositions need are already scored and only the two new stems are
computed. Neither casefix name is a key in `OMEGA`, which is why both calls name their
conditions and reference explicitly rather than going through the omega path, and
`--results_path` is what keeps `results/heldout_decomp_val.json` untouched.

Both calls will warn that `checkpoint_ladder` is stale. That is expected and unrelated to
this run: the ladder is copied from `results/rlsf_select_w3_*.json`, which predate the
fingerprint and need their own GPU pass to refresh.